In [0]:
# Create Players table view
spark.sql("""
CREATE OR REPLACE VIEW players_view AS
SELECT
  player_id,
  group_id
FROM VALUES
  (15, 1),
  (25, 1),
  (30, 1),
  (45, 1),
  (10, 2),
  (35, 2),
  (50, 2),
  (20, 3),
  (40, 3)
AS players(player_id, group_id)
""")

# Create Matches table view
spark.sql("""
CREATE OR REPLACE VIEW matches_view AS
SELECT
  match_id,
  first_player,
  second_player,
  first_score,
  second_score
FROM VALUES
  (1, 15, 45, 3, 0),
  (2, 30, 25, 1, 2),
  (3, 30, 15, 2, 0),
  (4, 40, 20, 5, 2),
  (5, 35, 50, 1, 1)
AS matches(match_id, first_player, second_player, first_score, second_score)
""")

In [0]:
spark.sql("""
    with scores_table as (
        select match_id, first_player as player, first_score as score from matches_view
        union all       
        select match_id, second_player as player, second_score as score from matches_view
    ), 
    grouped_scores as (
    select s.*, p.group_id from scores_table s join players_view p on s.player = p.player_id 
    ),
    scores as (
    select player, group_id, sum(score) as sc from grouped_scores
    group by group_id, player), 
    ranked as (
    select group_id, player, rank() over(partition by group_id order by sc desc, player) rnk from scores )

    select group_id, player from ranked where rnk  = 1

""").display()

In [0]:
spark.sql("""
    WITH scores_table AS (
        SELECT first_player AS player_id, first_score AS score FROM matches_view
        UNION ALL       
        SELECT second_player AS player_id, second_score AS score FROM matches_view
    ), 
    scores AS (
        -- Calculate total score per player before joining to make the join faster
        SELECT player_id, SUM(score) AS total_score 
        FROM scores_table
        GROUP BY player_id
    ), 
    ranked AS (
        SELECT 
            p.group_id, 
            s.player_id, 
            RANK() OVER(PARTITION BY p.group_id ORDER BY s.total_score DESC, s.player_id ASC) AS rnk 
        FROM scores s
        JOIN players_view p ON s.player_id = p.player_id 
    )
    SELECT group_id, player_id 
    FROM ranked 
    WHERE rnk = 1
""").display()